# DIVE : Tabular ML, NLP & Deep Learning Unified Platform

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aman-i1/DIVE/blob/main/examples/colab_quickstart.ipynb)

**DIVE** is a production-grade machine learning platform combining:
- **Tabular ML**: 17-category data health audits (`dive doctor`), resource-aware AutoML zoos (`dive train`), temporal leak detection, probability calibration, and failure slice discovery.
- **AutoNLP**: Diagnostic audits (`dive nlp audit`), multi-model benchmarking (TF-IDF + embeddings + tree/linear models), and zero-shot text classification (`dive nlp zero-shot`).
- **Deep Learning**: Modular PyTorch vision & audio adapters (`dive.dl`), auto-device detection (CUDA/MPS/CPU), early stopping, and LR schedulers.
- **MLOps & Governance**: Model registry, automated promotion gates, PSI/KS data & concept drift detection, and reproducible experiment replay.


## 1. Installation & Environment Verification

Install DIVE directly from GitHub with NLP and serving dependencies.

In [ ]:
# Clone the repository or install directly from GitHub
!git clone -q https://github.com/Aman-i1/DIVE.git
%cd DIVE
!pip install -q -e ".[nlp,serving]"


In [ ]:
# Verify CLI version and environment dependencies
!dive --version
!dive deps


## 2. ML Doctor Diagnostic Audit & Production Readiness Score

`dive doctor` runs a 17-category readiness audit to detect target leakage, entity contamination, duplicate leakage, uninformative columns, and computes a transparent Production Readiness Score (0-100).

In [ ]:
!dive doctor examples/sample.csv --target diagnosis


## 3. Resource-Aware Tabular AutoML Zoo Training

Train an ensemble of models with adaptive memory bounds, time budget, temporal validation (`--time-column`), probability calibration, and explainability.

In [ ]:
!dive train --data examples/sample.csv --target diagnosis --time-column scan_date --mode fast --output /content/out


In [ ]:
# Generate batch predictions
!dive predict --model /content/out/model.pkl --data examples/sample.csv --output /content/predictions.csv
!head -n 5 /content/predictions.csv


## 4. AutoNLP: Zero-Shot Classification & NLP AutoML

DIVE features first-class NLP capabilities including instant zero-shot classification and automated NLP pipeline benchmarking.

In [ ]:
# Instant zero-shot classification without prior model training
!dive nlp zero-shot "Critical payment gateway failure during customer checkout" --labels "billing,technical issue,account management,feedback"


In [ ]:
# Create a quick text classification sample dataset
import pandas as pd

df_nlp = pd.DataFrame({
    "text": [
        "The model training speed is exceptionally fast and accurate",
        "Terrible experience, the service timed out and failed repeatedly",
        "Comprehensive documentation with clear, working code examples",
        "Fatal crash occurred during data export, completely broken",
        "Superb customer support team resolved my issue within minutes",
        "App freezes intermittently on startup, needs urgent bugfix"
    ],
    "label": ["positive", "negative", "positive", "negative", "positive", "negative"]
})
df_nlp.to_csv("/content/sentiment.csv", index=False)
print(f"Created NLP dataset with {len(df_nlp)} samples.")


In [ ]:
# Run NLP health audit
!dive nlp audit /content/sentiment.csv --text-col text --label-col label


In [ ]:
# Train AutoNLP pipeline benchmarking multiple text models
!dive nlp train --data /content/sentiment.csv --text-col text --label-col label --output-dir /content/nlp_out


In [ ]:
# Score new text with calibrated class probabilities
!dive nlp predict --model /content/nlp_out/nlp_champion.pkl --text "The model training speed is exceptionally fast and accurate" --proba


## 5. Deep Learning & Vision Capabilities

DIVE's deep learning module (`dive.dl`) provides modular PyTorch adapters for image classification, audio processing, multimodal data handling, auto-hardware detection (CUDA/Apple Silicon MPS/CPU), and training callbacks.

In [ ]:
from dive.dl import (
    TorchDataModule,
    EarlyStopping,
    LRSchedulerManager,
    get_adapter,
    get_available_adapters,
)

print("Available Deep Learning Adapters in DIVE:")
for adapter_name in get_available_adapters():
    print(f"  * {adapter_name}")


In [ ]:
# Initialize Vision Image Classification Adapter
vision_adapter = get_adapter(
    "image_classification",
    num_classes=10,
    in_channels=3,
    image_size=224,
    learning_rate=1e-3,
)
print(f"Vision Adapter Architecture: {vision_adapter.model.__class__.__name__}")
print(f"Active Compute Device: {vision_adapter.device}")


## 6. Model Governance & Production Drift Monitoring

Register artifacts into the local model registry, enforce automated promotion gate checks, and monitor PSI / KS-test drift between reference and production batches.

In [ ]:
!dive models register /content/out/model.pkl --name cancer_classifier --stage candidate
!dive models list
!dive models promote cancer_classifier v1 production


In [ ]:
# Production data & prediction distribution drift monitoring
!dive drift --ref examples/sample.csv --curr examples/sample.csv


## 7. Python API Integration

In addition to the CLI, DIVE exposes high-level Python abstractions for interactive and scripted workflows.

In [ ]:
import pandas as pd
from dive import Dive, DiveDoctor

df = pd.read_csv('examples/sample.csv')

# 1. Run diagnostic audit
doctor = DiveDoctor(target='diagnosis')
report = doctor.analyze(df)
print(f"Data Health Score: {report.readiness_score}/100")
print(f"Issues Detected: {len(report.issues)}")

# 2. Train AutoML pipeline with cross-validation
pipeline = Dive(target='diagnosis', mode='fast')
pipeline.fit(df)
print(pipeline.leaderboard())
